

---
Требуется train_ds_good.csv




In [ ]:
!pip install torch
!pip install transformers
!pip install peft
!pip install datasets
!pip install -U bitsandbytes
!pip install accelerate
!pip install tqdm
!pip install evaluate
!pip install --upgrade gupload

   ╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/363.4 MB 31.0 MB/s eta 0:00:12Downloading nvidia_cublas_cu12-12.4.5.8-py3-none-manylinux2014_x86_64.whl (363.4 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.5 MB/s eta 0:00:00
   ╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/664.8 MB 55.7 MB/s eta 0:00:12Downloading nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl (664.8 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from accelerate import Accelerator
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, AdamW, get_scheduler, DataCollatorForLanguageModeling
import torch
import random
from datasets import load_dataset, Dataset
from peft import get_peft_model, prepare_model_for_kbit_training, LoraConfig
from huggingface_hub import login
import gc
from tqdm.notebook import tqdm
import evaluate
import re
import pandas as pd
import ast
import pandas as pd
from google.colab import drive
import numpy as np

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
TOKEN = None
LORA_DIMENSION_RANK = 32
LORA_ALPHA = 16
LORA_MODULES = ["q_proj", "v_proj"]
QUANT_TYPE="bf16"
BATCH_SIZE=1
NP_SAVE_PATH='pretrained_tensors'
MAX_LEN_NPZ=25

In [ ]:
class Config:
  def __init__(self):
    lora_dimension_rank = LORA_DIMENSION_RANK #из оригинала
    alpha_parameter_scaling = LORA_ALPHA
    self.peft_config = LoraConfig(lora_alpha=LORA_ALPHA, inference_mode=False, r=LORA_DIMENSION_RANK, bias = "none", task_type="CAUSAL_LM", target_modules=LORA_MODULES)
    self.bits_and_bytes_config = BitsAndBytesConfig(load_in_16bit=True,
                                 bnb_16bit_quant_type=QUANT_TYPE,
                                 bnb_16bit_compute_dtype=torch.float16,
                                 bnb_16bit_use_double_quant=True) #в оригинале используем квантизацию в 16, nf
config = Config()



Unused kwargs: ['load_in_16bit', 'bnb_16bit_quant_type', 'bnb_16bit_compute_dtype', 'bnb_16bit_use_double_quant']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.


In [ ]:
model_name = "facebook/opt-1.3b"
login(token=TOKEN)# -2

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name,ignore_mismatched_sizes=False,
                                          # quantization_config=config.bits_and_bytes_config,
                                          device_map="auto"
                                          )
tokenizer.max_len=256
tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/653 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

In [ ]:
pretrained_model = AutoModelForCausalLM.from_pretrained(model_name,ignore_mismatched_sizes=False,
                                                      #  quantization_config=config.bits_and_bytes_config,
                                                        output_hidden_states=False
                                                       )
# pretrained_model = prepare_model_for_kbit_training(pretrained_model)
# pretrained_model = get_peft_model(pretrained_model, config.peft_config)

pytorch_model.bin:   0%|          | 0.00/2.63G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [ ]:
batch_size = BATCH_SIZE

In [ ]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
# Whether or not to use masked language modeling. If set to False, the labels are the same as the inputs with the padding tokens ignored(-100)
#Кажется, мы так и хотим, только чистый инференс

In [ ]:
good_df_tr = pd.read_csv('train_ds_good.csv', sep='|')
good_df_tr['input_ids'] = good_df_tr['input_ids'].apply(lambda x: ast.literal_eval(x))
good_df_tr['attention_mask'] = good_df_tr['attention_mask'].apply(lambda x: ast.literal_eval(x))
good_train_ds = Dataset.from_pandas(good_df_tr)

In [ ]:
good_dataloader_pr_mod = torch.utils.data.DataLoader(
        good_train_ds, batch_size=batch_size, pin_memory=True, num_workers=1, shuffle=False, collate_fn=data_collator
    )
gc.collect()

311

In [ ]:
pretrained_model.to(device)

OPTForCausalLM(
  (model): OPTModel(
    (decoder): OPTDecoder(
      (embed_tokens): Embedding(50272, 2048, padding_idx=1)
      (embed_positions): OPTLearnedPositionalEmbedding(2050, 2048)
      (final_layer_norm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
      (layers): ModuleList(
        (0-23): 24 x OPTDecoderLayer(
          (self_attn): OPTSdpaAttention(
            (k_proj): Linear(in_features=2048, out_features=2048, bias=True)
            (v_proj): Linear(in_features=2048, out_features=2048, bias=True)
            (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
            (out_proj): Linear(in_features=2048, out_features=2048, bias=True)
          )
          (activation_fn): ReLU()
          (self_attn_layer_norm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
          (fc1): Linear(in_features=2048, out_features=8192, bias=True)
          (fc2): Linear(in_features=8192, out_features=2048, bias=True)
          (final_layer_norm): La

In [ ]:
result_r = []
cnt=0
for _, batch in tqdm(enumerate(good_dataloader_pr_mod),  total=len(good_dataloader_pr_mod),
                                                                           desc ="pretrained_outputs"):


  if len(result_r) >= MAX_LEN_NPZ:
    with open(NP_SAVE_PATH, 'ab') as f:
      save_path = f"{NP_SAVE_PATH}_{cnt}"
      np.savez(save_path, result_r, allow_pickle=False)
      cnt+=1

    # np.savez()
    result_r = []
    gc.collect()


  input_ids = torch.transpose(torch.cat(list(map(lambda el: el.unsqueeze(1), batch["input_ids"])), dim=1),0,1).to(device)
  attention_mask = torch.transpose(torch.cat(list(map(lambda el: el.unsqueeze(1), batch["attention_mask"])), dim=1),0,1).to(device)

  with torch.no_grad():
    out_logits = pretrained_model(input_ids, attention_mask = attention_mask).logits
    sf = torch.nn.functional.softmax(out_logits, dim=-1)
    prob_p = sf.chunk(sf.size(0), dim=0)
    prob_p = list(map(lambda el: el.squeeze(0),prob_p))

  for i in range(len(prob_p)):
    result_r.append(prob_p[i].detach().cpu().numpy())

pretrained_outputs:   0%|          | 0/75 [00:00<?, ?it/s]

In [ ]:
# total_cnt = 10

In [ ]:
# loaded_data = np.load(f"{NP_SAVE_PATH}_0.npz")
# # print(len(loaded_data["arr_0"][0][0]))
# loaded_data_1 = np.load(f"{NP_SAVE_PATH}_1.npz")
# # print(len(loaded_data["arr_0"]))

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# print(len(loaded_data_1["arr_0"]))

52


In [ ]:
len(good_dataloader_pr_mod)

75

In [ ]:
for i in range(cnt):
  print(i)
  NEW_SAVE_PATH=f"{NP_SAVE_PATH}_{i}.npz"
  !cp $NEW_SAVE_PATH /content/drive/MyDrive

0
1


In [ ]:
# del good_dataloader_pr_mod
# del pretrained_model
# gc.collect()

In [ ]:
# result_r = np.array(result_r)
# gc.collect()

In [ ]:
# result_r[1:]

In [ ]:
# np.savez(NP_SAVE_PATH, result_r)
# #6 минут на пределе памяти - это работает, не эффективно, но стабильно, pickle показывает еще более худшие результаты

In [ ]:
# applied_path_1 = "/content/drive/MyDrive/" + f"{NP_SAVE_PATH}_6.npz"
# loaded_data_1 = np.load(applied_path_1)

In [ ]:
# applied_path = "/content/drive/MyDrive/" + f"{NP_SAVE_PATH}_2.npz"
# loaded_data = np.load(applied_path)

In [ ]:
# loaded_data["arr_0"][0]

In [ ]:
# len(loaded_data_1["arr_0"])

In [ ]:
# type(loaded_data_1["arr_0"])

In [ ]:
# loaded_data_2 = np.stack([loaded_data["arr_0"], loaded_data_1["arr_0"]])

In [ ]:
# len(loaded_data_2)

In [ ]:
# !cp $NP_SAVE_PATH /content/drive/MyDrive

Сохранение файла на диск
#################################################